In [ ]:
import re
from datetime import datetime
from pathlib import Path

from ultralytics import YOLO

model_path = "yolo11l.pt"
model = YOLO(model_path)

# 폴더명: YYMMDD_모델명_testN (예: 260716_yolo11l_test3)
# N은 project 폴더 안에서 같은 날짜+모델의 기존 test 폴더 중 가장 큰 번호 + 1
model_name = Path(model_path).stem
project_dir = "runs"
date_str = datetime.now().strftime("%y%m%d")

existing_tests = [
    int(m.group(1))
    for p in Path(project_dir).glob(f"{date_str}_{model_name}_test*")
    if (m := re.search(r"_test(\d+)$", p.name))
]
next_test = max(existing_tests, default=0) + 1
run_name = f"{date_str}_{model_name}_test{next_test}"

model.train(
    data="/home/workstation/ai_cctv/dataset/data.yaml",
    epochs=50,
    imgsz=960,
    batch=-1,              # GPU 메모리에 맞춰 자동 설정 (권장)
    device=0,
    workers=16,
    patience=30,

    cache=True,            # RAM(125GB) 활용
    amp=True,              # Mixed Precision 사용
    cos_lr=True,           # Cosine Learning Rate
    close_mosaic=10,       # 마지막 10 epoch Mosaic 끄기
    save=True,
    plots=True,

    project=project_dir,
    name=run_name
)